# NLP Sentiment Classification on Customer Reviews (TF-IDF)

**Goal:** Clean customer-review text, convert it into TF-IDF features, train Multinomial Naive Bayes and Logistic Regression models, evaluate them, and classify custom review text.

> **Dataset note:** The 10 records supplied in the task are included below so the notebook runs immediately. The sample is very small and contains one neutral review. For the requested **positive vs. negative** classifier, neutral records are excluded from binary model training. Replace or extend the sample with a larger labeled dataset before drawing conclusions about model quality.

In [ ]:
# If needed, install dependencies in the notebook environment:
# %pip install pandas scikit-learn matplotlib nltk

In [ ]:
import re
import html
import warnings
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 120)

## 1. Load the review dataset

The notebook first tries the supplied CSV URL. If it is unavailable (for example, because the notebook has no internet access), it uses the exact 10 records provided in the task.

In [ ]:
DATA_URL = "https://www.datheon.in/datasets/sentiment_reviews.csv"

sample_csv = """review_id,review_text,sentiment
1,"The product quality is exceptional, exceeded all my expectations!",positive
2,"Terrible customer service and the item arrived completely damaged.",negative
3,"Average performance, decent for the price point but nothing special.",neutral
4,"Absolutely love this! Easy to setup, robust, and works flawlessly.",positive
5,"Stopped working after two days. Extremely disappointed with the purchase.",negative
6,"Customer support helped resolve my setup question within ten minutes.",positive
7,"Delivery was slow and package was missing one adapter cable.",negative
8,"Clean interface and intuitive controls. Highly recommended for students.",positive
9,"Not worth the money, plenty of better alternatives available.",negative
10,"Great value and durable build. Would definitely purchase again.",positive
"""

try:
    df = pd.read_csv(DATA_URL)
    print(f"Loaded dataset from: {DATA_URL}")
except Exception as exc:
    print("Could not fetch remote CSV; using the supplied 10-row sample.")
    print("Download error:", str(exc)[:250])
    df = pd.read_csv(StringIO(sample_csv))

# Normalize column names and identify likely text/label columns.
df.columns = [str(c).strip().lower() for c in df.columns]
text_candidates = ["review_text", "text", "review", "content"]
label_candidates = ["sentiment", "label", "polarity", "target"]
text_col = next((c for c in text_candidates if c in df.columns), None)
label_col = next((c for c in label_candidates if c in df.columns), None)

if text_col is None or label_col is None:
    raise ValueError(
        f"Could not identify text and sentiment columns. Found: {list(df.columns)}. "
        "Rename the columns to review_text and sentiment."
    )

df = df[[text_col, label_col]].rename(
    columns={text_col: "review_text", label_col: "sentiment"}
)
df = df.dropna(subset=["review_text", "sentiment"]).copy()
df["review_text"] = df["review_text"].astype(str)
df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()

print("Dataset shape:", df.shape)
display(df)
print("\nSentiment counts:")
display(df["sentiment"].value_counts().to_frame("count"))

## 2. Text preprocessing

Preprocessing performs HTML entity/tag removal, lowercasing, URL removal, punctuation/digit cleanup, whitespace normalization, and stopword removal. A lightweight suffix-based stem-like normalization is intentionally avoided; this keeps the notebook dependency-light. If you need true lemmatization, the optional NLTK cell below shows how to add it.

In [ ]:
def clean_text(text):
    """Clean one review string for TF-IDF feature extraction."""
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)          # HTML tags
    text = re.sub(r"https?://\\S+|www\\.\\S+", " ", text)  # URLs
    text = text.lower()
    text = re.sub(r"[^a-z\\s]", " ", text)       # punctuation, digits, symbols
    tokens = [word for word in text.split() if word not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

df["clean_text"] = df["review_text"].apply(clean_text)
display(df[["review_text", "clean_text", "sentiment"]])

In [ ]:
# Optional: true WordNet lemmatization with NLTK.
# Run this cell only if you want lemmatization and have internet access
# for the first-time NLTK resource download.
#
# import nltk
# nltk.download("wordnet")
# nltk.download("omw-1.4")
# from nltk.stem import WordNetLemmatizer
# lemmatizer = WordNetLemmatizer()
# def lemmatize_clean_text(text):
#     return " ".join(lemmatizer.lemmatize(token) for token in text.split())
# df["clean_text"] = df["clean_text"].apply(lemmatize_clean_text)

## 3. Prepare a binary positive/negative dataset

The supplied sample has one `neutral` row. Since the requested classifier is positive vs. negative, neutral rows are left out of model training. If your downloaded CSV uses different label spellings, normalize them in the mapping below.

In [ ]:
label_map = {
    "positive": "positive", "pos": "positive", "1": "positive",
    "negative": "negative", "neg": "negative", "0": "negative",
    "neutral": "neutral"
}
df["sentiment"] = df["sentiment"].map(lambda x: label_map.get(x, x))

binary_df = df[df["sentiment"].isin(["positive", "negative"])].copy()
if binary_df["sentiment"].nunique() != 2:
    raise ValueError("Training requires both positive and negative labels.")

X = binary_df["clean_text"]
y = binary_df["sentiment"]

print("Binary dataset size:", len(binary_df))
print("Excluded neutral/other rows:", len(df) - len(binary_df))
display(binary_df[["review_text", "sentiment"]])
print("\nClass counts:")
display(y.value_counts().to_frame("count"))

if y.value_counts().min() < 2:
    raise ValueError("Each class needs at least two examples for a stratified train/test split.")

## 4. TF-IDF features and train/test split

`TfidfVectorizer` is fitted on training text only (inside each model pipeline), preventing vocabulary/IDF leakage from the test set. With only nine labeled binary examples in the supplied sample, evaluation metrics are highly unstable; a larger dataset is strongly recommended.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.33,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train records: {len(X_train)} | Test records: {len(X_test)}")
print("\nTrain label counts:")
display(y_train.value_counts().to_frame("count"))
print("\nTest label counts:")
display(y_test.value_counts().to_frame("count"))

## 5. Train Multinomial Naive Bayes and Logistic Regression

Both models use the same TF-IDF settings for a fair baseline comparison. `max_features=5000` caps the vocabulary size.

In [ ]:
models = {
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ("classifier", MultinomialNB())
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ])
}

results = []
trained_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results.append({"Model": model_name, "Accuracy": accuracy})
    trained_models[model_name] = model

    print(f"\\n===== {model_name} =====")
    print(f"Accuracy: {accuracy:.3f}")
    print(classification_report(y_test, predictions, zero_division=0))

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df)

In [ ]:
# Plot accuracy benchmark (interpret cautiously for this tiny sample).
plt.figure(figsize=(8, 4))
plt.bar(results_df["Model"], results_df["Accuracy"])
plt.ylim(0, 1.05)
plt.ylabel("Test accuracy")
plt.title("Model accuracy comparison")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for both models.
for model_name, model in trained_models.items():
    predictions = model.predict(X_test)
    cm = confusion_matrix(y_test, predictions, labels=["negative", "positive"])
    display_labels = ["negative", "positive"]
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix — {model_name}")
    plt.tight_layout()
    plt.show()

## 6. Select a model and classify custom reviews

The default selection is the model with the highest held-out accuracy. On a tiny sample, this is only a demonstration—not a reliable model-selection procedure. The helper accepts one sentence or a list of review strings.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Selected demonstration model:", best_model_name)

def predict_sentiment(text, model=best_model):
    """Return predicted polarity and class probabilities for custom review text."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please provide a non-empty review string.")

    cleaned = clean_text(text)
    prediction = model.predict([cleaned])[0]
    output = {"review": text, "cleaned_text": cleaned, "predicted_sentiment": prediction}

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba([cleaned])[0]
        output["probabilities"] = dict(zip(model.classes_, probabilities.round(4)))

    return output

# Try a few sample inputs:
test_phrases = [
    "I love this product, it works perfectly and is excellent quality.",
    "The item broke immediately and customer service was awful.",
    "Great value, easy to use, and I would buy it again."
]

for phrase in test_phrases:
    print(predict_sentiment(phrase))


In [ ]:
# Interactive input when running the notebook locally.
# Enter 'quit' to stop.
while True:
    user_review = input("\\nEnter a customer review (or 'quit'): ").strip()
    if user_review.lower() in {"quit", "exit", "q"}:
        print("Finished.")
        break
    if not user_review:
        print("Please enter some text.")
        continue
    print(predict_sentiment(user_review))

## 7. Conclusion and recommended improvements

- The notebook demonstrates cleaning, TF-IDF vectorization, Multinomial Naive Bayes, Logistic Regression, evaluation, and custom predictions.
- The supplied dataset has only 10 rows (9 positive/negative after excluding neutral), so the reported test accuracy is not a meaningful estimate of real-world performance.
- For a credible benchmark, use hundreds or thousands of labeled reviews, preserve a separate test set, and consider stratified cross-validation and hyperparameter tuning.
- If neutral sentiment is important, train a three-class classifier (`negative`, `neutral`, `positive`) instead of filtering neutral records.
- For reproducibility, record the dataset source/version and keep the same label definitions across training and inference.